# LongPort Data Explorer
**Goal:** Confirm what historical data LongPort provides — timeframes, history depth, and instrument types.

Paste your demo credentials in the cell below, then run all cells top to bottom.

In [1]:
# ── STEP 1: Your LongPort Demo Credentials ──────────────────────────────────
APP_KEY      = "6e73506603d52be23b36cc884ca6fd8b"
APP_SECRET   = "66785f824db9072cf288d52ea0a1ca6649f3fddb61d4f0a01534277ce2500c58"
ACCESS_TOKEN = "m_eyJhbGciOiJSUzI1NiIsImtpZCI6ImQ5YWRiMGIxYTdlNzYxNzEiLCJ0eXAiOiJKV1QifQ.eyJpc3MiOiJsb25nYnJpZGdlIiwic3ViIjoiYWNjZXNzX3Rva2VuIiwiZXhwIjoxNzgzMzE2MTcxLCJpYXQiOjE3NzU1NDAxNzEsImFrIjoiNmU3MzUwNjYwM2Q1MmJlMjNiMzZjYzg4NGNhNmZkOGIiLCJhYWlkIjoyMTMxNTMyNCwiYWMiOiJsYl9wYXBlcnRyYWRpbmciLCJtaWQiOjI2ODYxNDA3LCJzaWQiOiJodThMbkc4aU9kdDVDbHVzc21zYlh3PT0iLCJibCI6MiwidWwiOjAsImlrIjoibGJfcGFwZXJ0cmFkaW5nXzIxMzE1MzI0In0.MmWpwyTMDJq4KDF4sCtSgxRUbRlpTjZPINZGDsZxx9JZa-kOIADW2Lz69PpAf0BZ8ZxU8SZFVAoNZyKnViw_rgO8W5Q6gKcdmwxPkWP9fGl57TVVBK0n4rj3bxnuwA5Fwv1NJVnLx7tRoE4drRSTHIMZDJFCTztGoF5wlV8Jrj9N2fYp5YK97MLATufEzdb4CILpWn74yTHWoWccjUKurOmaCu7nhvAbqCbpttS5SWUEIY_1RyC_HYzdxjnA3vl0XFCYB5MbsZLux8YhiUOCWuTLS_qTgilQMamNFK20h_L3WZlKFfTRxeQ41ycj1Y-rePCiLAr2UNX1SfHB7kqMstBXQ6v-qhAw52E7TdePfDI1EW51YCLepnQvu-AXeb2nlr9GHEV8KRMAyxOoIZg3dlfLBFue4VfxlO-J8Rku3BetdaqPTp0v_aAtos2effXbaM1zKrkyMZk3IO26fpw1t8ZqGHZhkrw90pG5LATcWb95CtdsdVmjgo4cXTU18whHARl6gCO6PfovHirXr3755dPxv8cKRVgeHXECwr43_aAzWV03DSbxBVL_335W4bI5nO8NJc1PpQsuvaFq8Md6wL8AsBxPFoks6R1rzuqA6tFyoApQaZ4DC-bfbnOdpg7Yon4AJpjGxOnOBBYxVVSpQuvxvHpytzCEkOpdrDiTKyQ"

In [2]:
# ── STEP 2: Connect ──────────────────────────────────────────────────────────
from longport.openapi import Config, QuoteContext, Period, AdjustType
from datetime import datetime
import pandas as pd

cfg = Config(app_key=APP_KEY, app_secret=APP_SECRET, access_token=ACCESS_TOKEN)
ctx = QuoteContext(cfg)
print("✅ Connected to LongPort")

✅ Connected to LongPort


In [3]:
# ── STEP 3: Helper — fetch candles and summarise ─────────────────────────────
def probe(symbol, period, label, count=1000):
    try:
        candles = ctx.candlesticks(symbol, period, count, AdjustType.ForwardAdjust)
        if not candles:
            return {"symbol": symbol, "period": label, "bars": 0,
                    "oldest": "—", "newest": "—", "note": "empty"}
        dates = [c.timestamp for c in candles]
        oldest = min(dates)
        newest = max(dates)
        return {
            "symbol":  symbol,
            "period":  label,
            "bars":    len(candles),
            "oldest":  oldest.strftime("%Y-%m-%d") if hasattr(oldest, 'strftime') else str(oldest)[:10],
            "newest":  newest.strftime("%Y-%m-%d") if hasattr(newest, 'strftime') else str(newest)[:10],
            "note":    "ok"
        }
    except Exception as e:
        return {"symbol": symbol, "period": label, "bars": 0,
                "oldest": "—", "newest": "—", "note": str(e)[:80]}

print("Helper ready")

Helper ready


In [4]:
# ── STEP 4: HK Stocks — How deep is the history? ─────────────────────────────
print("Testing HK stocks across all timeframes...\n")

hk_symbol = "700.HK"   # Tencent — very liquid, good test case

periods = [
    (Period.Min_1,  "1min"),
    (Period.Min_5,  "5min"),
    (Period.Min_15, "15min"),
    (Period.Min_30, "30min"),
    (Period.Min_60, "60min"),
    (Period.Day,    "1day"),
    (Period.Week,   "1week"),
]

rows = [probe(hk_symbol, p, label) for p, label in periods]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

Testing HK stocks across all timeframes...

symbol period  bars     oldest     newest note
700.HK   1min  1000 2026-04-10 2026-04-15   ok
700.HK   5min  1000 2026-03-20 2026-04-15   ok
700.HK  15min  1000 2026-02-04 2026-04-15   ok
700.HK  30min  1000 2025-12-04 2026-04-15   ok
700.HK  60min  1000 2025-09-08 2026-04-15   ok
700.HK   1day  1000 2022-03-16 2026-04-15   ok
700.HK  1week  1000 2007-02-19 2026-04-13   ok


In [5]:
# ── STEP 5: US Stocks — Same test ────────────────────────────────────────────
print("Testing US stocks...\n")

us_symbol = "AAPL.US"

rows_us = [probe(us_symbol, p, label) for p, label in periods]
df_us = pd.DataFrame(rows_us)
print(df_us.to_string(index=False))

Testing US stocks...

 symbol period  bars     oldest     newest note
AAPL.US   1min  1000 2026-04-11 2026-04-15   ok
AAPL.US   5min  1000 2026-03-26 2026-04-15   ok
AAPL.US  15min  1000 2026-02-19 2026-04-15   ok
AAPL.US  30min  1000 2025-12-20 2026-04-15   ok
AAPL.US  60min  1000 2025-09-17 2026-04-15   ok
AAPL.US   1day  1000 2022-04-19 2026-04-14   ok
AAPL.US  1week  1000 2007-02-19 2026-04-13   ok


In [6]:
# ── STEP 6: Multiple HK symbols ──────────────────────────────────────────────
print("Testing multiple HK symbols (daily only)...\n")

hk_symbols = [
    "700.HK",   # Tencent
    "9988.HK",  # Alibaba
    "0005.HK",  # HSBC
    "2318.HK",  # Ping An
    "0388.HK",  # HKEX
    "1299.HK",  # AIA
]

rows_multi = [probe(s, Period.Day, "1day") for s in hk_symbols]
df_multi = pd.DataFrame(rows_multi)
print(df_multi.to_string(index=False))

Testing multiple HK symbols (daily only)...

 symbol period  bars     oldest     newest note
 700.HK   1day  1000 2022-03-16 2026-04-15   ok
9988.HK   1day  1000 2022-03-16 2026-04-15   ok
0005.HK   1day  1000 2022-03-16 2026-04-15   ok
2318.HK   1day  1000 2022-03-16 2026-04-15   ok
0388.HK   1day  1000 2022-03-16 2026-04-15   ok
1299.HK   1day  1000 2022-03-16 2026-04-15   ok


In [7]:
# ── STEP 7: US symbols ───────────────────────────────────────────────────────
print("Testing multiple US symbols (daily only)...\n")

us_symbols = [
    "AAPL.US",
    "TSLA.US",
    "NVDA.US",
    "SPY.US",
    "QQQ.US",
    "MSFT.US",
]

rows_us_multi = [probe(s, Period.Day, "1day") for s in us_symbols]
df_us_multi = pd.DataFrame(rows_us_multi)
print(df_us_multi.to_string(index=False))

Testing multiple US symbols (daily only)...

 symbol period  bars     oldest     newest note
AAPL.US   1day  1000 2022-04-19 2026-04-14   ok
TSLA.US   1day  1000 2022-04-19 2026-04-14   ok
NVDA.US   1day  1000 2022-04-19 2026-04-14   ok
 SPY.US   1day  1000 2022-04-19 2026-04-14   ok
 QQQ.US   1day  1000 2022-04-19 2026-04-14   ok
MSFT.US   1day  1000 2022-04-19 2026-04-14   ok


In [8]:
# ── STEP 8: HK Warrants ───────────────────────────────────────────────────────
# HK warrants have codes like 12345.HK
# These are common student instruments — test if LP provides data
print("Testing HK Warrants/CBBCs (daily)...\n")

warrant_symbols = [
    "21001.HK",  # Example warrant — may or may not exist
    "52000.HK",  # CBBC example
]

rows_w = [probe(s, Period.Day, "1day", count=200) for s in warrant_symbols]
df_w = pd.DataFrame(rows_w)
print(df_w.to_string(index=False))
print("\nNote: Warrants have short lifespans — fewer bars is normal")

Testing HK Warrants/CBBCs (daily)...

  symbol period  bars     oldest     newest                                                                             note
21001.HK   1day   131 2025-09-29 2026-04-15                                                                               ok
52000.HK   1day     0          —          — OpenApiException: (kind=ErrorKind.OpenApi, code=301600, trace_id=) invalid symbo

Note: Warrants have short lifespans — fewer bars is normal


In [9]:
# ── STEP 9: ETFs ─────────────────────────────────────────────────────────────
print("Testing ETFs...\n")

etf_symbols = [
    "2800.HK",   # Tracker Fund of HK (HSI ETF)
    "3032.HK",   # CSOP Hang Seng TECH ETF
    "GLD.US",    # Gold ETF
    "TLT.US",    # 20yr Treasury ETF
]

rows_etf = [probe(s, Period.Day, "1day") for s in etf_symbols]
df_etf = pd.DataFrame(rows_etf)
print(df_etf.to_string(index=False))

Testing ETFs...

 symbol period  bars     oldest     newest note
2800.HK   1day  1000 2022-03-16 2026-04-15   ok
3032.HK   1day  1000 2022-03-16 2026-04-15   ok
 GLD.US   1day  1000 2022-04-19 2026-04-14   ok
 TLT.US   1day  1000 2022-04-19 2026-04-14   ok


In [10]:
# ── STEP 10: Check what static info LP provides per symbol ───────────────────
print("Checking static info for 700.HK (lot size, currency, type)...\n")

try:
    details = ctx.static_info(["700.HK", "AAPL.US", "2800.HK"])
    for d in details:
        print(f"{d.symbol:12s} | name: {d.name_en:30s} | lot: {d.lot_size:6} | currency: {d.currency} | type: {d.security_type}")
except Exception as e:
    print(f"static_info error: {e}")

Checking static info for 700.HK (lot size, currency, type)...

static_info error: 'builtins.SecurityStaticInfo' object has no attribute 'security_type'


In [11]:
# ── STEP 11: Summary ─────────────────────────────────────────────────────────
print("="*60)
print("SUMMARY")
print("="*60)
print("\n700.HK timeframe depth:")
print(df[["period","bars","oldest","newest","note"]].to_string(index=False))
print("\nAAPL.US timeframe depth:")
print(df_us[["period","bars","oldest","newest","note"]].to_string(index=False))
print("\n✅ If 'bars' > 0 and 'note' = ok → LongPort can feed the backtester")
print("⚠️  If 'bars' = 0 or error → fallback to Yahoo for that timeframe")

SUMMARY

700.HK timeframe depth:
period  bars     oldest     newest note
  1min  1000 2026-04-10 2026-04-15   ok
  5min  1000 2026-03-20 2026-04-15   ok
 15min  1000 2026-02-04 2026-04-15   ok
 30min  1000 2025-12-04 2026-04-15   ok
 60min  1000 2025-09-08 2026-04-15   ok
  1day  1000 2022-03-16 2026-04-15   ok
 1week  1000 2007-02-19 2026-04-13   ok

AAPL.US timeframe depth:
period  bars     oldest     newest note
  1min  1000 2026-04-11 2026-04-15   ok
  5min  1000 2026-03-26 2026-04-15   ok
 15min  1000 2026-02-19 2026-04-15   ok
 30min  1000 2025-12-20 2026-04-15   ok
 60min  1000 2025-09-17 2026-04-15   ok
  1day  1000 2022-04-19 2026-04-14   ok
 1week  1000 2007-02-19 2026-04-13   ok

✅ If 'bars' > 0 and 'note' = ok → LongPort can feed the backtester
⚠️  If 'bars' = 0 or error → fallback to Yahoo for that timeframe


In [12]:
# Test if LP has more than 1000 bars — push the count limit
from longport.openapi import Period, AdjustType
import pandas as pd

def probe(ctx, symbol, period, label, count):
    try:
        candles = ctx.candlesticks(symbol, period, count, AdjustType.ForwardAdjust)
        if not candles:
            return {"requested": count, "returned": 0, "oldest": "—", "newest": "—", "note": "empty"}
        dates = [c.timestamp for c in candles]
        return {
            "requested": count,
            "returned":  len(candles),
            "oldest":    str(min(dates))[:10],
            "newest":    str(max(dates))[:10],
            "note":      "⚠️ MAX HIT" if len(candles) < count else "ok"
        }
    except Exception as e:
        return {"requested": count, "returned": 0, "oldest": "—", "newest": "—", "note": str(e)[:100]}

counts = [500, 1000, 2000, 5000, 10000]
rows = [probe(ctx, "700.HK", Period.Day, "1day", c) for c in counts]
print(pd.DataFrame(rows).to_string(index=False))

 requested  returned     oldest     newest                                                                                       note
       500       500 2024-03-28 2026-04-15                                                                                         ok
      1000      1000 2022-03-16 2026-04-15                                                                                         ok
      2000         0          —          — OpenApiException: (kind=ErrorKind.OpenApi, code=301607, trace_id=) request too many klines
      5000         0          —          — OpenApiException: (kind=ErrorKind.OpenApi, code=301607, trace_id=) request too many klines
     10000         0          —          — OpenApiException: (kind=ErrorKind.OpenApi, code=301607, trace_id=) request too many klines


In [13]:
# Test pagination — can we stitch multiple 1000-bar calls together?
# LongPort candlesticks() doesn't have an offset param, 
# but let's check if there's a date-range version of the API

# First check what methods are available on the context
methods = [m for m in dir(ctx) if not m.startswith('_')]
print("Available QuoteContext methods:")
for m in methods:
    print(" ", m)

Available QuoteContext methods:
  brokers
  calc_indexes
  candlesticks
  capital_distribution
  capital_flow
  create_watchlist_group
  delete_watchlist_group
  depth
  history_candlesticks_by_date
  history_candlesticks_by_offset
  history_market_temperature
  intraday
  market_temperature
  member_id
  option_chain_expiry_date_list
  option_chain_info_by_date
  option_quote
  participants
  quote
  quote_level
  quote_package_details
  realtime_brokers
  realtime_candlesticks
  realtime_depth
  realtime_quote
  realtime_trades
  security_list
  set_on_brokers
  set_on_candlestick
  set_on_depth
  set_on_quote
  set_on_trades
  static_info
  subscribe
  subscribe_candlesticks
  subscriptions
  trades
  trading_days
  trading_session
  unsubscribe
  unsubscribe_candlesticks
  update_watchlist_group
  warrant_issuers
  warrant_list
  warrant_quote
  watchlist


In [14]:
# Explore history_candlesticks_by_date and history_candlesticks_by_offset
# Check their signatures first
import inspect
print("history_candlesticks_by_date signature:")
print(inspect.signature(ctx.history_candlesticks_by_date))
print()
print("history_candlesticks_by_offset signature:")
print(inspect.signature(ctx.history_candlesticks_by_offset))

history_candlesticks_by_date signature:
(symbol, period, adjust_type, start=None, end=None, trade_sessions=Ellipsis)

history_candlesticks_by_offset signature:
(symbol, period, adjust_type, forward, count, time=None, trade_sessions=Ellipsis)


In [15]:
# Test history_candlesticks_by_offset — paginate backwards to get max history
from longport.openapi import Period, AdjustType
from datetime import datetime

def fetch_all_history(ctx, symbol, period, label):
    """Paginate backwards in 1000-bar chunks until no more data."""
    all_candles = []
    anchor_time = None  # start from today, walk backwards
    max_pages = 20      # safety limit — 20 x 1000 = 20,000 bars max
    
    for page in range(max_pages):
        try:
            batch = ctx.history_candlesticks_by_offset(
                symbol, period, AdjustType.ForwardAdjust,
                forward=False,   # go backwards
                count=1000,
                time=anchor_time
            )
        except Exception as e:
            print(f"  Page {page+1} error: {e}")
            break
            
        if not batch:
            print(f"  Page {page+1}: empty — stopping")
            break
        
        all_candles = list(batch) + all_candles  # prepend older data
        oldest = str(batch[0].timestamp)[:10]
        newest = str(batch[-1].timestamp)[:10]
        print(f"  Page {page+1}: {len(batch)} bars | {oldest} → {newest}")
        
        if len(batch) < 1000:
            print(f"  Page {page+1}: less than 1000 bars returned — reached the beginning")
            break
        
        # Set anchor to oldest bar timestamp for next page
        anchor_time = batch[0].timestamp
    
    return all_candles

print("=" * 55)
print("700.HK — Daily — paginating backwards")
print("=" * 55)
candles_day = fetch_all_history(ctx, "700.HK", Period.Day, "1day")
print(f"\nTotal daily bars: {len(candles_day)}")
if candles_day:
    print(f"Full range: {str(candles_day[0].timestamp)[:10]} → {str(candles_day[-1].timestamp)[:10]}")

700.HK — Daily — paginating backwards
  Page 1: 1000 bars | 2022-03-16 → 2026-04-15
  Page 2: 1000 bars | 2018-02-23 → 2022-03-16
  Page 3: 1000 bars | 2014-02-05 → 2018-02-23
  Page 4: 1000 bars | 2010-01-15 → 2014-02-05
  Page 5: 1000 bars | 2005-12-28 → 2010-01-15
  Page 6: 382 bars | 2004-06-16 → 2005-12-28
  Page 6: less than 1000 bars returned — reached the beginning

Total daily bars: 5382
Full range: 2004-06-16 → 2026-04-15


In [16]:
# Test pagination for 1min and 5min
print("=" * 55)
print("700.HK — 1min — paginating backwards")
print("=" * 55)
candles_1min = fetch_all_history(ctx, "700.HK", Period.Min_1, "1min")
print(f"\nTotal 1min bars: {len(candles_1min)}")
if candles_1min:
    print(f"Full range: {str(candles_1min[0].timestamp)[:19]} → {str(candles_1min[-1].timestamp)[:19]}")

print()
print("=" * 55)
print("700.HK — 5min — paginating backwards")
print("=" * 55)
candles_5min = fetch_all_history(ctx, "700.HK", Period.Min_5, "5min")
print(f"\nTotal 5min bars: {len(candles_5min)}")
if candles_5min:
    print(f"Full range: {str(candles_5min[0].timestamp)[:19]} → {str(candles_5min[-1].timestamp)[:19]}")

700.HK — 1min — paginating backwards
  Page 1: 1000 bars | 2026-04-10 → 2026-04-15
  Page 2: 1000 bars | 2026-04-02 → 2026-04-10
  Page 3: 1000 bars | 2026-03-30 → 2026-04-02
  Page 4: 1000 bars | 2026-03-25 → 2026-03-30
  Page 5: 1000 bars | 2026-03-20 → 2026-03-25
  Page 6: 1000 bars | 2026-03-17 → 2026-03-20
  Page 7: 1000 bars | 2026-03-12 → 2026-03-17
  Page 8: 1000 bars | 2026-03-09 → 2026-03-12
  Page 9: 1000 bars | 2026-03-04 → 2026-03-09
  Page 10: 1000 bars | 2026-02-27 → 2026-03-04
  Page 11: 1000 bars | 2026-02-24 → 2026-02-27
  Page 12: 1000 bars | 2026-02-13 → 2026-02-24
  Page 13: 1000 bars | 2026-02-10 → 2026-02-13
  Page 14: 1000 bars | 2026-02-05 → 2026-02-10
  Page 15: 1000 bars | 2026-02-02 → 2026-02-05
  Page 16: 1000 bars | 2026-01-28 → 2026-02-02
  Page 17: 1000 bars | 2026-01-23 → 2026-01-28
  Page 18: 1000 bars | 2026-01-20 → 2026-01-23
  Page 19: 1000 bars | 2026-01-15 → 2026-01-20
  Page 20: 1000 bars | 2026-01-12 → 2026-01-15

Total 1min bars: 20000
Full ran

In [17]:
# Test 15min and 60min depth + close the context cleanly
print("=" * 55)
print("700.HK — 15min — paginating backwards")
print("=" * 55)
candles_15min = fetch_all_history(ctx, "700.HK", Period.Min_15, "15min")
print(f"\nTotal 15min bars: {len(candles_15min)}")
if candles_15min:
    print(f"Full range: {str(candles_15min[0].timestamp)[:19]} → {str(candles_15min[-1].timestamp)[:19]}")

print()
print("=" * 55)
print("700.HK — 60min — paginating backwards")  
print("=" * 55)
candles_60min = fetch_all_history(ctx, "700.HK", Period.Min_60, "60min")
print(f"\nTotal 60min bars: {len(candles_60min)}")
if candles_60min:
    print(f"Full range: {str(candles_60min[0].timestamp)[:19]} → {str(candles_60min[-1].timestamp)[:19]}")

# Close context cleanly
ctx.close()
print("\n✅ QuoteContext closed cleanly")

700.HK — 15min — paginating backwards
  Page 1: 1000 bars | 2026-02-04 → 2026-04-15
  Page 2: 1000 bars | 2025-12-01 → 2026-02-04
  Page 3: 1000 bars | 2025-09-25 → 2025-12-01
  Page 4: 1000 bars | 2025-07-28 → 2025-09-25
  Page 5: 1000 bars | 2025-05-26 → 2025-07-28
  Page 6: 1000 bars | 2025-03-19 → 2025-05-26
  Page 7: 1000 bars | 2025-01-13 → 2025-03-19
  Page 8: 1000 bars | 2024-11-06 → 2025-01-13
  Page 9: 1000 bars | 2024-09-02 → 2024-11-06
  Page 10: 1000 bars | 2024-07-02 → 2024-09-02
  Page 11: 1000 bars | 2024-04-26 → 2024-07-02
  Page 12: 400 bars | 2024-04-02 → 2024-04-26
  Page 12: less than 1000 bars returned — reached the beginning

Total 15min bars: 11400
Full range: 2024-04-02 09:30:00 → 2026-04-15 13:00:00

700.HK — 60min — paginating backwards
  Page 1: 1000 bars | 2025-09-08 → 2026-04-15
  Page 2: 1000 bars | 2025-02-11 → 2025-09-08
  Page 3: 1000 bars | 2024-07-09 → 2025-02-11
  Page 4: 454 bars | 2024-04-02 → 2024-07-09
  Page 4: less than 1000 bars returned — re

AttributeError: 'builtins.QuoteContext' object has no attribute 'close'

In [18]:
# Confirm full 60min depth with more pages
ctx2 = QuoteContext(cfg)  # reopen since we closed ctx

def fetch_all_history_verbose(ctx, symbol, period, label, max_pages=50):
    all_candles = []
    anchor_time = None
    for page in range(max_pages):
        try:
            batch = ctx.history_candlesticks_by_offset(
                symbol, period, AdjustType.ForwardAdjust,
                forward=False, count=1000, time=anchor_time
            )
        except Exception as e:
            print(f"  Page {page+1} error: {e}")
            break
        if not batch:
            print(f"  Page {page+1}: empty — stopping")
            break
        all_candles = list(batch) + all_candles
        oldest = str(batch[0].timestamp)[:10]
        newest = str(batch[-1].timestamp)[:10]
        print(f"  Page {page+1}: {len(batch)} bars | {oldest} → {newest}")
        if len(batch) < 1000:
            print(f"  Reached the beginning")
            break
        anchor_time = batch[0].timestamp
    return all_candles

print("=" * 55)
print("700.HK — 60min — full depth (max 50 pages)")
print("=" * 55)
candles = fetch_all_history_verbose(ctx2, "700.HK", Period.Min_60, "60min", max_pages=50)
print(f"\nTotal 60min bars: {len(candles)}")
if candles:
    print(f"Full range: {str(candles[0].timestamp)[:19]} → {str(candles[-1].timestamp)[:19]}")

ctx2.close()
print("✅ Context closed")

700.HK — 60min — full depth (max 50 pages)
  Page 1: 1000 bars | 2025-09-08 → 2026-04-15
  Page 2: 1000 bars | 2025-02-11 → 2025-09-08
  Page 3: 1000 bars | 2024-07-09 → 2025-02-11
  Page 4: 454 bars | 2024-04-02 → 2024-07-09
  Reached the beginning

Total 60min bars: 3454
Full range: 2024-04-02 09:30:00 → 2026-04-15 13:00:00


AttributeError: 'builtins.QuoteContext' object has no attribute 'close'

In [19]:
# Find the correct way to release QuoteContext
methods = [m for m in dir(ctx2) if not m.startswith('_')]
print("All methods:", methods)

# Also test — does it release when we delete it?
print("\nTrying del ctx2...")
del ctx2
print("✅ del ctx2 succeeded — garbage collection handles cleanup")

# Test creating a fresh one works fine after del
print("\nTesting fresh connection after del...")
ctx3 = QuoteContext(cfg)
test = ctx3.candlesticks("700.HK", Period.Day, 5, AdjustType.ForwardAdjust)
print(f"✅ Fresh context works — got {len(test)} bars")
del ctx3
print("✅ del ctx3 — done")

All methods: ['brokers', 'calc_indexes', 'candlesticks', 'capital_distribution', 'capital_flow', 'create_watchlist_group', 'delete_watchlist_group', 'depth', 'history_candlesticks_by_date', 'history_candlesticks_by_offset', 'history_market_temperature', 'intraday', 'market_temperature', 'member_id', 'option_chain_expiry_date_list', 'option_chain_info_by_date', 'option_quote', 'participants', 'quote', 'quote_level', 'quote_package_details', 'realtime_brokers', 'realtime_candlesticks', 'realtime_depth', 'realtime_quote', 'realtime_trades', 'security_list', 'set_on_brokers', 'set_on_candlestick', 'set_on_depth', 'set_on_quote', 'set_on_trades', 'static_info', 'subscribe', 'subscribe_candlesticks', 'subscriptions', 'trades', 'trading_days', 'trading_session', 'unsubscribe', 'unsubscribe_candlesticks', 'update_watchlist_group', 'warrant_issuers', 'warrant_list', 'warrant_quote', 'watchlist']

Trying del ctx2...
✅ del ctx2 succeeded — garbage collection handles cleanup

Testing fresh connect

In [21]:
# Test LongPort security search capabilities
ctx4 = QuoteContext(cfg)

# Test 1: static_info lookup by known symbols
print("Test 1: static_info for known symbols")
try:
    results = ctx4.static_info(["AAPL.US", "700.HK", "9988.HK", "2800.HK"])
    for r in results:
        print(vars(r))
        print("---")
except Exception as e:
    print(f"Error: {e}")

Test 1: static_info for known symbols
<built-in method __dict__ of builtins.SecurityStaticInfo object at 0x0000021FC52721F0>
---
<built-in method __dict__ of builtins.SecurityStaticInfo object at 0x0000021FC6D2D6B0>
---
<built-in method __dict__ of builtins.SecurityStaticInfo object at 0x0000021FC6D2D7D0>
---
<built-in method __dict__ of builtins.SecurityStaticInfo object at 0x0000021FC6D2D8F0>
---


In [22]:
# Inspect SecurityStaticInfo fields properly
results = ctx4.static_info(["AAPL.US", "700.HK", "9988.HK", "2800.HK"])
for r in results:
    print(dir(r))
    print("---")
    break  # just look at first one

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'board', 'bps', 'circulating_shares', 'currency', 'dividend_yield', 'eps', 'eps_ttm', 'exchange', 'hk_shares', 'lot_size', 'name_cn', 'name_en', 'name_hk', 'stock_derivatives', 'symbol', 'total_shares']
---


In [23]:
# Read all fields from static_info results
results = ctx4.static_info(["AAPL.US", "700.HK", "9988.HK", "2800.HK", "TSLA.US", "NVDA.US"])
for r in results:
    print(f"symbol={r.symbol} | name_en={r.name_en} | lot_size={r.lot_size} | "
          f"currency={r.currency} | exchange={r.exchange} | board={r.board} | "
          f"stock_derivatives={r.stock_derivatives}")
    print()

# Also test security_list — does LP have a search endpoint?
print("="*55)
print("Testing security_list...")
try:
    sec_list = ctx4.security_list("US", "Equity")
    print(f"security_list returned {len(sec_list)} results")
    if sec_list:
        r = sec_list[0]
        print(f"First result fields: {dir(r)}")
        print(f"First result: symbol={r.symbol} name={r.name_en if hasattr(r,'name_en') else '?'}")
except Exception as e:
    print(f"security_list error: {e}")

del ctx4

symbol=AAPL.US | name_en=Apple Inc. | lot_size=1 | currency=USD | exchange=NASD | board=SecurityBoard.Unknown | stock_derivatives=[DerivativeType.Option]

symbol=700.HK | name_en=TENCENT | lot_size=100 | currency=HKD | exchange=SEHK | board=SecurityBoard.HKEquity | stock_derivatives=[DerivativeType.Warrant]

symbol=9988.HK | name_en=BABA-W | lot_size=100 | currency=HKD | exchange=SEHK | board=SecurityBoard.HKEquity | stock_derivatives=[DerivativeType.Warrant]

symbol=2800.HK | name_en=TRACKER FUND | lot_size=500 | currency=HKD | exchange=SEHK | board=SecurityBoard.HKEquity | stock_derivatives=[DerivativeType.Warrant]

symbol=TSLA.US | name_en=Tesla, Inc. | lot_size=1 | currency=USD | exchange=NASD | board=SecurityBoard.Unknown | stock_derivatives=[DerivativeType.Option]

symbol=NVDA.US | name_en=NVIDIA Corporation | lot_size=1 | currency=USD | exchange=NASD | board=SecurityBoard.Unknown | stock_derivatives=[DerivativeType.Option]

Testing security_list...
security_list error: argument 

In [24]:
ctx5 = QuoteContext(cfg)

# Fix security_list with proper Market enum
from longport.openapi import Market
print("Market enum values:")
print([m for m in dir(Market) if not m.startswith('_')])

# Test warrant_list — useful for HK warrant search
print("\nTesting warrant_list for 700.HK...")
try:
    warrants = ctx5.warrant_list(
        symbol="700.HK",
        sort_by=0,
        sort_order=0,
    )
    print(f"warrant_list returned: {type(warrants)}")
    if warrants and hasattr(warrants, '__iter__'):
        items = list(warrants)
        print(f"Count: {len(items)}")
        if items:
            print(f"First warrant fields: {[a for a in dir(items[0]) if not a.startswith('_')]}")
            print(f"First warrant: symbol={items[0].symbol}")
except Exception as e:
    print(f"warrant_list error: {e}")

# Test static_info with a fuzzy guess — what if user types partial?
print("\nTest: static_info with wrong symbol format...")
try:
    r = ctx5.static_info(["AAPL", "700", "TENCENT"])
    for x in r:
        print(f"  {x.symbol} | {x.name_en}")
except Exception as e:
    print(f"Partial symbol error: {e}")

del ctx5

Market enum values:
['CN', 'Crypto', 'HK', 'SG', 'US', 'Unknown']

Testing warrant_list for 700.HK...
warrant_list error: argument 'sort_by': 'int' object cannot be cast as 'WarrantSortBy'

Test: static_info with wrong symbol format...


In [25]:
ctx6 = QuoteContext(cfg)
from longport.openapi import Market, WarrantSortBy, SortOrderType

# Test 1: security_list with Market enum
print("Test 1: security_list with Market enum")
try:
    sec_list = ctx6.security_list(Market.US, "Equity")
    print(f"US Equity list: {len(sec_list)} results")
    if sec_list:
        print(f"First 3: {[(r.symbol, r.name_en) for r in sec_list[:3]]}")
except Exception as e:
    print(f"security_list error: {e}")

# Test 2: warrant_list with proper enums
print("\nTest 2: warrant_list for 700.HK")
try:
    warrants = ctx6.warrant_list(
        symbol="700.HK",
        sort_by=WarrantSortBy.LastDone,
        sort_order=SortOrderType.Descending,
    )
    items = list(warrants) if warrants else []
    print(f"Warrants for 700.HK: {len(items)}")
    if items:
        print(f"Fields: {[a for a in dir(items[0]) if not a.startswith('_')]}")
        print(f"First 3: {[(w.symbol, w.name if hasattr(w,'name') else '?') for w in items[:3]]}")
except Exception as e:
    print(f"warrant_list error: {e}")

# Test 3: static_info partial — does LP auto-correct?
print("\nTest 3: partial symbol lookup")
try:
    r = ctx6.static_info(["AAPL", "700", "TENCENT"])
    for x in r:
        print(f"  found: {x.symbol} | {x.name_en}")
except Exception as e:
    print(f"Partial lookup error (expected): {e}")

del ctx6

Test 1: security_list with Market enum
security_list error: argument 'category': 'str' object cannot be cast as 'SecurityListCategory'

Test 2: warrant_list for 700.HK
Warrants for 700.HK: 668
Fields: ['balance_point', 'call_price', 'change_rate', 'change_value', 'conversion_ratio', 'delta', 'effective_leverage', 'expiry_date', 'implied_volatility', 'itm_otm', 'last_done', 'leverage_ratio', 'lower_strike_price', 'name', 'outstanding_qty', 'outstanding_ratio', 'premium', 'status', 'strike_price', 'symbol', 'to_call_price', 'turnover', 'upper_strike_price', 'volume', 'warrant_type']
First 3: [('24687.HK', 'JPTENCT@EP2606A'), ('24760.HK', 'UBTENCT@EP2606B'), ('25228.HK', 'GJTENCT@EP2606B')]

Test 3: partial symbol lookup
